# Contour CSVs: stitching, RANSAC and registration error from pre-analysed contour centres

Same analysis as `stitch_viewer.ipynb`, but the contact centres come from **pre-analysed contour
CSV files** instead of the SEM images and `.oas` files: one CSV per image, one row per contact,
columns `DesignX_Add`, `DesignY_Add` (design centre) and `SEMX_Add`, `SEMY_Add` (SEM centre), in
**global mask coordinates, nm**. Design and SEM on the same row are the same contact, so they are
not matched again.

1. **Read** every `*.csv` in `CSV_DIR` (`read_contour_folder`). Rows with a missing value are left
   out and counted; files without any contact are listed and left out.
2. **Stitching**, translation per tile, as for the images: the SEM centres (`stitch_tiles`) and,
   the same way, the design centres, which can be offset between files. There is no metadata
   CSV, so each tile's box is the bounding box of its design centres (`tile_boxes_from_points`):
   two tiles overlap where both report contacts.
3. **Errors** per placement (`paired_errors`, raw SEM − design, no affine), then one entry per
   physical contact (`merge_observations`).
4. **RANSAC** global affine, **moving-window** affine and **row pitch** on the merged contacts.
5. **Analysis window** (`AnalysisWindow`, section 8): RANSAC monitor, registration error (RANSAC
   and moving window, each with the mean error per row **without affine** as a reference),
   moving-window tuner, row pitch. There is no stitching viewer: it needs the SEM images.

The window blocks this notebook until it is closed. Positions in µm, errors in nm.

In [ ]:
CSV_DIR = r"path\to\csv_folder"
MAX_MATCH_NM = 25.0         # SEM overlaps: pair contacts closer than this, the largest B - A stage offset (< pitch / 2)
DESIGN_MAX_MATCH_NM = 25.0  # design overlaps: the largest offset between neighbouring files (< pitch / 2)
OUTLIER_FACTOR = 3.0        # overlap outlier if residual > this x the median residual
MIN_MATCHED = 5             # pairs with fewer matched contacts are not used (reported below)
GROUP_RADIUS_NM = 5.0       # observations from different tiles whose stitched design positions are this close = one contact
SPREAD_FLAG_NM = 2.0        # flag merged contacts whose SEM observations differ by more than this
USE_OPENGL = True           # set False if the viewer window stays black
RANSAC_PLACEMENT = "mean"   # RANSAC on the merged contacts of this placement: "mean" (stitched) or "first"
RANSAC_THRESHOLD_NM = 0.5   # RANSAC inlier if |G(SEM) - design| < this
RANSAC_SEED = 0             # RANSAC random samples are reproducible for a given seed
RANSAC_DELAY_MS = 300       # RANSAC monitor: pause between steps
ROW_GAP_NM = 10.0           # registration error per row: a new row starts where sorted design y jumps by more than this
MOVING_WINDOW_UM = 40.0     # moving-window affine: window height along y
MOVING_STEP_UM = 5.0        # moving-window affine: the window moves up this much each step

In [ ]:
import numpy as np
import pandas as pd
import pyqtgraph as pg

from affine_ransac.fitting.moving_window import moving_window_affine
from affine_ransac.fitting.ransac import ransac_affine
from affine_ransac.geometry.affine import report_terms
from affine_ransac.io.contour_csv import read_contour_folder
from affine_ransac.pipeline import stitch_tiles, tile_boxes_from_points
from affine_ransac.registration import error_summary, merge_observations, paired_errors, row_means, row_pitch
from affine_ransac.stitching import placement_corrections
from affine_ransac.view_analysis import AnalysisWindow

## 1. Read the contour CSVs

One row per file: contacts used, rows left out (missing value), the area covered by its design
centres (µm) and its mean SEM − design before stitching (nm).

In [ ]:
all_tiles = read_contour_folder(CSV_DIR)
empty = [t.name for t in all_tiles if len(t.design_nm) == 0]
if empty:
    print(f"!! {len(empty)} file(s) without any contact, left out: {empty}")
tiles = [t for t in all_tiles if len(t.design_nm)]
tile_ids = [t.name for t in tiles]
print(f"{len(tiles)} of {len(all_tiles)} files used, {sum(len(t.design_nm) for t in tiles)} contacts, "
      f"{sum(t.dropped for t in all_tiles)} rows left out (missing value)")

pd.DataFrame({
    "tile": tile_ids,
    "contacts": [len(t.design_nm) for t in tiles],
    "rows_left_out": [t.dropped for t in tiles],
    "x_min_um": [t.design_nm[:, 0].min() / 1000 for t in tiles], "x_max_um": [t.design_nm[:, 0].max() / 1000 for t in tiles],
    "y_min_um": [t.design_nm[:, 1].min() / 1000 for t in tiles], "y_max_um": [t.design_nm[:, 1].max() / 1000 for t in tiles],
    "mean_dx_nm": [(t.sem_nm - t.design_nm)[:, 0].mean() for t in tiles],
    "mean_dy_nm": [(t.sem_nm - t.design_nm)[:, 1].mean() for t in tiles],
}).round(4)

## 2. Stitching: SEM centres and design centres

In [ ]:
sem_points = [t.sem_nm for t in tiles]
design_points = [t.design_nm for t in tiles]
centers, sizes = tile_boxes_from_points(design_points)  # bounding box of each tile's design centres
stitch = stitch_tiles(sem_points, centers, sizes, MAX_MATCH_NM, OUTLIER_FACTOR, MIN_MATCHED)
design_stitch = stitch_tiles(design_points, centers, sizes, DESIGN_MAX_MATCH_NM, OUTLIER_FACTOR, MIN_MATCHED,
                             label="design")

for name, result in (("SEM", stitch), ("Design", design_stitch)):
    failed = [r for r in result.rejected if r.failed]
    print(f"{name}: {len(result.pairs)} overlapping pairs used, {len(failed)} not matched, "
          f"{len(result.rejected) - len(failed)} skipped (fewer than {MIN_MATCHED} contacts in the overlap); "
          f"corrections |x| max {np.nanmax(np.abs(result.corrections[:, 0])):.2f} nm, "
          f"|y| max {np.nanmax(np.abs(result.corrections[:, 1])):.2f} nm")
    if failed or len(result.unplaced):
        print(f"!! {name.upper()} STITCHING INCOMPLETE. Tiles not stitched: {[tile_ids[k] for k in result.unplaced]}")
        display(pd.DataFrame([{"tile_a": tile_ids[r.i], "tile_b": tile_ids[r.j], "contacts_in_overlap": r.in_box,
                               "matched": r.matched} for r in failed]))

pd.DataFrame({
    "tile": tile_ids,
    "sem_correction_x_nm": stitch.corrections[:, 0], "sem_correction_y_nm": stitch.corrections[:, 1],
    "design_correction_x_nm": design_stitch.corrections[:, 0], "design_correction_y_nm": design_stitch.corrections[:, 1],
}).round(2)

## 3. Errors per placement (raw: SEM − design, no affine removed)

Each CSV row is one observation; a contact in an overlap is observed once per tile (2 in a strip,
up to 4 in a corner). `merge_observations` turns them into one entry per physical contact:
observations from different tiles whose **stitched design** positions are within
`GROUP_RADIUS_NM` are averaged, and their **spread** (largest distance between the SEM
observations) is kept. The table compares the placements: nominal (CSV coordinates as given),
stitched (corrections average 0), stitched with the first tile fixed.

In [ ]:
observations, merged = {}, {}
for placement, (sem_corrections, design_corrections) in placement_corrections(stitch.corrections,
                                                                               design_stitch.corrections).items():
    observations[placement] = paired_errors(sem_points, design_points, sem_corrections, design_corrections)
    merged[placement] = merge_observations(observations[placement], design_stitch.corrections, GROUP_RADIUS_NM)

for placement, e in observations.items():
    if e.skipped:
        print(f"!! {placement}: {len(e.skipped)} tile(s) NOT MEASURED (no error computed):")
        display(pd.DataFrame([{"tile": tile_ids[k], "reason": reason} for k, reason in e.skipped.items()]))

m = merged["mean"]
counts = pd.Series(m.count).value_counts().sort_index()
print(f"{len(observations['mean'].tile)} observations -> {len(m.count)} contacts; "
      f"observations per contact: {({int(k): int(v) for k, v in counts.items()})}")
overlap = m.count > 1
if overlap.any():
    print(f"Spread of merged overlap contacts (stitched): median {np.median(m.spread_nm[overlap]):.2f} nm, "
          f"max {m.spread_nm.max():.2f} nm; {(m.spread_nm > SPREAD_FLAG_NM).sum()} above {SPREAD_FLAG_NM} nm")
    worst = np.argsort(m.spread_nm)[::-1][:10]
    display(pd.DataFrame({
        "x_um": m.design_nm[worst, 0] / 1000, "y_um": m.design_nm[worst, 1] / 1000,
        "spread_nm": m.spread_nm[worst], "observations": m.count[worst],
        "tiles": [[tile_ids[k] for k in observations["mean"].tile[m.members[g]]] for g in worst],
    }).round(4))

pd.DataFrame({placement: error_summary(e.error_nm) for placement, e in merged.items()}).round(2)

## 4. RANSAC global affine (on the merged contacts)

`ransac_affine` finds one affine G for the whole field that maps the stitched SEM centres onto the
design centres: random 3-point samples, the model with the most inliers (residual <
`RANSAC_THRESHOLD_NM`), then a least-squares refit on its inliers, all relative to the design
centroid. Outliers are only left out of the fit; their residuals are kept. The live monitor is the
*RANSAC monitor* tab of the analysis window (section 8).

**Table** (correction G, SEM → design): Tx, Ty in nm; Mx, My in ppm (= nm per mm of distance);
rotation and orthogonality in degrees. *edge (nm)*: how far that term alone moves the furthest
contact, comparable to the threshold and the residuals.

In [ ]:
contacts = merged[RANSAC_PLACEMENT]
result = ransac_affine(contacts.sem_nm, contacts.design_nm, RANSAC_THRESHOLD_NM, seed=RANSAC_SEED)

print(f"{result.iterations} search iterations; best sample {result.best_sample.tolist()}")
print(f"Inliers: {result.inliers.sum()} / {len(result.inliers)} ({result.inliers.mean():.1%}), "
      f"reference (design centroid): x {result.reference[0] / 1000:.4f} um, y {result.reference[1] / 1000:.4f} um")
display(pd.DataFrame(report_terms(result.model, contacts.design_nm - result.reference),
                     columns=["term (correction G)", "value", "edge (nm)"]).set_index("term (correction G)"))
print("G = [[a, b, tx], [c, d, ty]] about the reference point:")
print(np.array2string(result.model[:2], precision=9, suppress_small=False))
pd.DataFrame({"inliers": error_summary(result.residuals[result.inliers]),
              "all contacts": error_summary(result.residuals)}).round(3)

## 5. Registration error after the RANSAC affine

The final registration error of every contact: its residual after G, G(SEM) − design, outliers
included. The table lists the mean dx, dy of each row of contacts (rows grouped by design y,
`ROW_GAP_NM`), after the affine and, for reference, of the stitched contacts without any affine
(SEM − design). Both are plotted in the *Registration* tabs (section 8).

In [ ]:
registration_error = result.residuals  # (N, 2) G(SEM) - design per merged contact, nm (outliers included)
row_y, row_mean, row_count = row_means(contacts.design_nm[:, 1], registration_error, ROW_GAP_NM)
_, raw_row_mean, _ = row_means(contacts.design_nm[:, 1], contacts.error_nm, ROW_GAP_NM)  # stitched, no affine
print(f"{len(row_y)} rows, {row_count.min()}..{row_count.max()} contacts per row")
pd.DataFrame({"row_y_um": (row_y - result.reference[1]) / 1000, "contacts": row_count,
              "dx_mean_nm": row_mean[:, 0], "dy_mean_nm": row_mean[:, 1],
              "no_affine_dx_mean_nm": raw_row_mean[:, 0], "no_affine_dy_mean_nm": raw_row_mean[:, 1]}).round(3)

## 6. Moving-window affine, compared with the RANSAC affine

A plain least-squares affine in a window `MOVING_WINDOW_UM` tall along y, moved up by
`MOVING_STEP_UM`; each contact is corrected by the window whose centre is nearest to its design y.
The cell compares the summaries and per-row means with the RANSAC result and lists the affine terms
of every window (correction G).

In [ ]:
moving = moving_window_affine(contacts.sem_nm, contacts.design_nm, MOVING_WINDOW_UM * 1000, MOVING_STEP_UM * 1000)
print(f"{len(moving.centres_nm)} windows, {moving.counts.min()}..{moving.counts.max()} contacts per window")
display(pd.DataFrame({"RANSAC inliers": error_summary(result.residuals[result.inliers]),
                      "RANSAC all contacts": error_summary(result.residuals),
                      "moving window all contacts": error_summary(moving.residuals)}).round(3))

window_terms = pd.DataFrame(
    [{label: value for label, value, _ in report_terms(model, contacts.design_nm[moving.window == k] - reference)}
     for k, (model, reference) in enumerate(zip(moving.models, moving.references))])
window_terms.insert(0, "contacts", moving.counts)
window_terms.insert(0, "centre_y_um", (moving.centres_nm - result.reference[1]) / 1000)
window_terms.round(6)

## 7. Row-to-row pitch along y

Rows grouped by design y (`ROW_GAP_NM`); the pitch is the spacing between neighbouring rows' mean y.
Three sets of the same merged contacts: the **design**, the **stitched SEM without any affine** and
the **SEM after the RANSAC affine**. Tile edges (dotted lines in the *Row pitch* tab) are the edges
of each tile's bounding box of contacts.

In [ ]:
pitch_sets = {"stitched, no affine": contacts.sem_nm, "after RANSAC affine": contacts.design_nm + result.residuals}
mid_y, design_pitch = row_pitch(contacts.design_nm[:, 1], contacts.design_nm[:, 1], ROW_GAP_NM)
pitch_table = pd.DataFrame({"row_y_um": (mid_y - result.reference[1]) / 1000, "design_nm": design_pitch})
for name, points in pitch_sets.items():
    pitch_table[f"{name} - design (nm)"] = row_pitch(contacts.design_nm[:, 1], points[:, 1], ROW_GAP_NM)[1] - design_pitch
display(pitch_table.describe().round(3))
worst = pitch_table.iloc[:, 2:].abs().max(axis=1).nlargest(10).index
display(pitch_table.loc[worst].round(3))
tile_edges_y = np.unique(np.round(np.concatenate([centers[:, 1] - sizes[:, 1] / 2, centers[:, 1] + sizes[:, 1] / 2])))

## 8. Analysis window (closes with the window)

All viewers in one window, one tab each:

- **RANSAC monitor**: press Run or Step to watch the fit of section 4 (same τ and seed).
- **Registration, RANSAC** / **Registration, moving window**: dx and dy error maps (jet, one dot
  per contact), the mean error per row after the affine and, below it, the reference: the mean
  error per row of the stitched contacts **without any affine**.
- **Moving-window tuner**: sliders for the window height and step; error maps, row means (solid:
  moving window, dashed: RANSAC) and the affine term chosen in the list per window.
- **Row pitch**: section 7.

In [ ]:
app = pg.mkQApp("AffineRANSAC analysis")
window = AnalysisWindow(contacts.sem_nm, contacts.design_nm, result, moving, MOVING_WINDOW_UM, MOVING_STEP_UM,
                        RANSAC_THRESHOLD_NM, RANSAC_SEED, RANSAC_DELAY_MS, ROW_GAP_NM, tile_edges_y,
                        use_opengl=USE_OPENGL)
window.show()
app.exec()
print(f"Tuner, last setting: window {window.tuner.window.value():g} um, step {window.tuner.step.value():g} um")